# Archon — the autonomous bookkeeper 🇬🇷
### Kaggle × Google AI Agents Intensive (Vibe Coding) — Capstone

Archon reads the documents a small business actually receives — **sales invoices,
purchase invoices, bank transactions, payroll** — **classifies** them, posts
**double-entry journal entries**, **reconciles** invoices against bank payments,
and rolls everything up into a **P&L, a cash view, and AR/AP**.

It's an AI **bookkeeper**, not a dashboard — and it keeps real books, so the
numbers are auditable. Built with **Google ADK + Gemini** on **GCP** (the GCP
member of the Archon family, which also runs on Nebius and Azure).

> The cell below inlines the whole engine so this notebook runs top-to-bottom
> with **zero setup**. The conversational ADK+Gemini agent is at the end (add a
> `GOOGLE_API_KEY` secret to run it).

In [ ]:
from __future__ import annotations

# ── archon/models.py ─────────────────────────────────────────
"""Archon (GCP) — domain models for an autonomous bookkeeping agent.

Archon ingests the heterogeneous documents a small business actually receives
— sales invoices, purchase invoices, bank transactions, payroll — classifies
them, posts double-entry journal entries, reconciles invoices against payments,
and rolls everything up into a P&L, a cash position, and AR/AP.

Payroll is just one document family here (a single payroll run is itself told by
several documents and several journal lines — net pay to the employee, EFKA to
the institution, withheld tax to the authority). Nothing about this is "the
payroll truth"; it's ordinary bookkeeping, done by an agent.
"""


from dataclasses import dataclass, field
from enum import Enum


class DocType(str, Enum):
    SALES_INVOICE = "sales_invoice"
    PURCHASE_INVOICE = "purchase_invoice"
    BANK_TRANSACTION = "bank_transaction"
    PAYROLL = "payroll"
    UNKNOWN = "unknown"


class Account(str, Enum):
    """A minimal chart of accounts (Greek SMB flavour, jurisdiction-pluggable)."""
    REVENUE = "Revenue"
    COGS = "Cost of Goods / Services"
    OPEX = "Operating Expenses"
    PAYROLL_EXPENSE = "Payroll Expense"            # gross + employer contributions
    VAT_OUTPUT = "VAT Payable (output)"
    VAT_INPUT = "VAT Receivable (input)"
    ACCOUNTS_RECEIVABLE = "Accounts Receivable"
    ACCOUNTS_PAYABLE = "Accounts Payable"
    BANK = "Bank"
    NET_PAY_PAYABLE = "Net Pay Payable"
    EFKA_PAYABLE = "EFKA / Social Insurance Payable"
    TAX_WITHHELD_PAYABLE = "Withheld Tax Payable"


@dataclass
class Document:
    """A classified, structured business document."""
    doc_type: DocType
    period: str                      # YYYY-MM
    counterparty: str | None = None
    document_number: str | None = None
    date: str | None = None
    currency: str = "EUR"
    # amounts (meaning depends on doc_type)
    net_amount: float | None = None      # ex-VAT for invoices; transfer amount for bank
    vat_amount: float | None = None
    gross_amount: float | None = None    # incl. VAT (invoices) / gross pay (payroll)
    # bank-specific
    direction: str | None = None         # "in" | "out"
    reference: str | None = None         # what the bank line references (invoice no., "ΜΙΣΘΟΔΟΣΙΑ")
    # payroll-specific (one fused run)
    employer_cost_total: float | None = None
    net_pay_total: float | None = None
    efka_total: float | None = None
    tax_withheld_total: float | None = None
    employee_count: int | None = None
    source_file: str | None = None


@dataclass
class JournalLine:
    account: Account
    debit: float = 0.0
    credit: float = 0.0


@dataclass
class JournalEntry:
    """A balanced double-entry posting derived from one document."""
    date: str | None
    period: str
    memo: str
    lines: list[JournalLine] = field(default_factory=list)
    source_doc: str | None = None

    @property
    def is_balanced(self) -> bool:
        return abs(sum(l.debit for l in self.lines) - sum(l.credit for l in self.lines)) < 0.01


@dataclass
class ReconciliationMatch:
    bank_ref: str
    matched_document: str | None
    amount: float
    matched: bool
    note: str


@dataclass
class ValidationResult:
    """Outcome of one cross-document consistency gate (R1–R4).

    The validator is the deterministic **eval gate**: books are only trusted
    when every rule that applies passes. A skipped rule (required documents
    absent) is reported as passed so it never blocks a partial month.
    """
    rule: str
    passed: bool
    severity: str          # "info" | "warning" | "error"
    message: str


@dataclass
class FinancialStatements:
    period: str
    revenue: float
    cogs: float
    opex: float
    payroll_expense: float
    net_profit: float
    cash_in: float
    cash_out: float
    net_cash: float
    accounts_receivable: float
    accounts_payable: float
    bank_balance_movement: float
    gross_margin_pct: float | None
    notes: list[str] = field(default_factory=list)


# ── archon/extract.py ────────────────────────────────────────
"""Classify + extract a business document into structured fields.

In production this tool sends a scanned PDF to **Gemini** (vision) on GCP; here
it parses OCR-style text blobs deterministically so the notebook runs with no
key. Greek diacritics are folded so unaccented patterns match accented text.
"""


import re
import unicodedata



_AMOUNT = r"([\d\.]+,\d{2})"


def _fold(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")


def _eur(text: str, label: str) -> float | None:
    # Non-greedy, same-line gap so a rate between label and amount
    # ("ΦΠΑ 24%: 1.200,00") is skipped to the first properly-formatted amount.
    m = re.search(r"(?:" + label + r")[^\n]*?" + _AMOUNT, _fold(text), re.IGNORECASE)
    if not m:
        return None
    return round(float(m.group(1).replace(".", "").replace(",", ".")), 2)


def _int(text: str, label: str) -> int | None:
    m = re.search(r"(?:" + label + r")[^\d]*(\d+)", _fold(text), re.IGNORECASE)
    return int(m.group(1)) if m else None


def _str(text: str, label: str) -> str | None:
    m = re.search(r"(?:" + label + r")\s*:?\s*(.+)", text, re.IGNORECASE)
    return m.group(1).strip() if m and m.group(1) else None


def classify(text: str) -> DocType:
    t = _fold(text).lower()
    if "τιμολογιο πωλησης" in t or "sales invoice" in t:
        return DocType.SALES_INVOICE
    if "τιμολογιο αγορας" in t or "purchase invoice" in t:
        return DocType.PURCHASE_INVOICE
    if "κινηση λογαριασμου" in t or "bank transaction" in t:
        return DocType.BANK_TRANSACTION
    if "μισθοδοσια" in t or "payroll" in t:
        return DocType.PAYROLL
    return DocType.UNKNOWN


def extract_document(text: str, source_file: str = "doc.txt", period: str = "2026-01") -> Document:
    dtype = classify(text)
    doc = Document(doc_type=dtype, period=period, source_file=source_file,
                   document_number=_str(text, r"Αριθμ\w+"), date=_str(text, r"Ημερομην\w+"))

    if dtype in (DocType.SALES_INVOICE, DocType.PURCHASE_INVOICE):
        doc.counterparty = _str(text, r"Πελ\w+") or _str(text, r"Προμηθευτ\w+")
        doc.net_amount = _eur(text, r"καθαρη αξια|net")
        doc.vat_amount = _eur(text, r"φπα")
        doc.gross_amount = _eur(text, r"συνολο|total")
    elif dtype == DocType.BANK_TRANSACTION:
        tf = _fold(text).lower()
        doc.direction = "in" if ("εισερχ" in tf or "(in)" in tf) else "out"
        doc.net_amount = _eur(text, r"ποσο|amount")
        doc.reference = _str(text, r"Αιτιολογ\w+|reference")
        doc.date = _str(text, r"Ημερομην\w+")
    elif dtype == DocType.PAYROLL:
        doc.gross_amount = _eur(text, r"μικτ\w+ αποδοχ\w+|gross")
        doc.net_pay_total = _eur(text, r"καθαρ\w+ πληρωτ\w+|net")
        doc.efka_total = _eur(text, r"εργοδοτικ\w+ εισφορ\w+|employer.*efka")
        doc.tax_withheld_total = _eur(text, r"παρακρατουμεν\w+ φορ\w+|fmy|withheld")
        doc.employer_cost_total = _eur(text, r"συνολικ\w+ εργοδοτικ\w+ κοστος|employer cost")
        doc.employee_count = _int(text, r"εργαζομεν\w+|employees")
    return doc


# ── archon/ledger.py ─────────────────────────────────────────
"""The bookkeeping core — deterministic double-entry posting, reconciliation,
and statement roll-up. The LLM never touches the math; it orchestrates this.

Every document becomes a balanced journal entry; bank lines are matched to the
invoice/payroll they settle; everything rolls up to a P&L, a cash view, and
AR/AP. Country-specific posting rules live here (this is the Greek/EFKA flavour;
swap this module to localise).
"""




Dr = lambda a, x: JournalLine(a, debit=round(x, 2))   # noqa: E731
Cr = lambda a, x: JournalLine(a, credit=round(x, 2))   # noqa: E731

_PAYROLL_KEYS = ("ΜΙΣΘΟΔ", "μισθοδ", "payroll", "PAYROLL")


class Ledger:
    def __init__(self, period: str, company: str | None = None):
        self.period = period
        self.company = company
        self.documents: list[Document] = []
        self.entries: list[JournalEntry] = []

    # ── posting ───────────────────────────────────────────────────────────────

    def add(self, doc: Document) -> JournalEntry:
        self.documents.append(doc)
        entry = self._post(doc)
        self.entries.append(entry)
        return entry

    def _post(self, doc: Document) -> JournalEntry:
        e = JournalEntry(date=doc.date, period=doc.period, memo="", source_doc=doc.source_file)

        if doc.doc_type == DocType.SALES_INVOICE:
            net = doc.net_amount or 0.0
            vat = doc.vat_amount or 0.0
            gross = doc.gross_amount or (net + vat)
            e.memo = f"Sales invoice {doc.document_number} to {doc.counterparty}"
            e.lines = [Dr(Account.ACCOUNTS_RECEIVABLE, gross),
                       Cr(Account.REVENUE, net)]
            if vat:
                e.lines.append(Cr(Account.VAT_OUTPUT, vat))

        elif doc.doc_type == DocType.PURCHASE_INVOICE:
            net = doc.net_amount or 0.0
            vat = doc.vat_amount or 0.0
            gross = doc.gross_amount or (net + vat)
            e.memo = f"Purchase invoice {doc.document_number} from {doc.counterparty}"
            e.lines = [Dr(Account.OPEX, net)]
            if vat:
                e.lines.append(Dr(Account.VAT_INPUT, vat))
            e.lines.append(Cr(Account.ACCOUNTS_PAYABLE, gross))

        elif doc.doc_type == DocType.BANK_TRANSACTION:
            amt = doc.net_amount or 0.0
            ref = doc.reference or ""
            if doc.direction == "in":
                e.memo = f"Receipt {ref}"
                e.lines = [Dr(Account.BANK, amt), Cr(Account.ACCOUNTS_RECEIVABLE, amt)]
            elif any(k in ref for k in _PAYROLL_KEYS):
                e.memo = f"Payroll net paid ({ref})"
                e.lines = [Dr(Account.NET_PAY_PAYABLE, amt), Cr(Account.BANK, amt)]
            else:
                e.memo = f"Payment {ref}"
                e.lines = [Dr(Account.ACCOUNTS_PAYABLE, amt), Cr(Account.BANK, amt)]

        elif doc.doc_type == DocType.PAYROLL:
            gross = doc.gross_amount or 0.0
            net = doc.net_pay_total or 0.0
            employer_efka = doc.efka_total or 0.0
            tax = doc.tax_withheld_total or 0.0
            employer_cost = doc.employer_cost_total or (gross + employer_efka)
            employee_efka = round(gross - net - tax, 2)        # implied employee deduction
            efka_payable = round(employer_efka + employee_efka, 2)
            e.memo = f"Payroll {doc.period} ({doc.employee_count} employees)"
            e.lines = [
                Dr(Account.PAYROLL_EXPENSE, employer_cost),
                Cr(Account.NET_PAY_PAYABLE, net),
                Cr(Account.EFKA_PAYABLE, efka_payable),
                Cr(Account.TAX_WITHHELD_PAYABLE, tax),
            ]
        else:
            e.memo = f"Unclassified document {doc.source_file}"
        return e

    # ── reconciliation ────────────────────────────────────────────────────────

    def reconcile(self) -> list[ReconciliationMatch]:
        matches: list[ReconciliationMatch] = []
        by_number = {d.document_number: d for d in self.documents if d.document_number}
        for d in self.documents:
            if d.doc_type != DocType.BANK_TRANSACTION:
                continue
            ref = d.reference or ""
            amt = d.net_amount or 0.0
            if any(k in ref for k in _PAYROLL_KEYS):
                payroll = next((x for x in self.documents if x.doc_type == DocType.PAYROLL), None)
                ok = payroll is not None and abs((payroll.net_pay_total or 0) - amt) < 0.01
                matches.append(ReconciliationMatch(ref, "payroll", amt, ok,
                               "matched payroll net" if ok else "no matching payroll net"))
            elif ref in by_number:
                inv = by_number[ref]
                ok = abs((inv.gross_amount or 0) - amt) < 0.01
                matches.append(ReconciliationMatch(ref, ref, amt, ok,
                               "matched invoice" if ok else "amount mismatch vs invoice"))
            else:
                matches.append(ReconciliationMatch(ref, None, amt, False, "no matching document"))
        return matches

    # ── statements ────────────────────────────────────────────────────────────

    def _bal(self) -> dict[Account, tuple[float, float]]:
        b: dict[Account, list[float]] = {}
        for e in self.entries:
            for ln in e.lines:
                d, c = b.setdefault(ln.account, [0.0, 0.0])
                b[ln.account] = [round(d + ln.debit, 2), round(c + ln.credit, 2)]
        return {k: (v[0], v[1]) for k, v in b.items()}

    def statements(self) -> FinancialStatements:
        b = self._bal()

        def dr(a):  # debit-minus-credit
            d, c = b.get(a, (0.0, 0.0))
            return round(d - c, 2)

        def cr(a):  # credit-minus-debit
            d, c = b.get(a, (0.0, 0.0))
            return round(c - d, 2)

        revenue = cr(Account.REVENUE)
        cogs = dr(Account.COGS)
        opex = dr(Account.OPEX)
        payroll = dr(Account.PAYROLL_EXPENSE)
        net_profit = round(revenue - cogs - opex - payroll, 2)

        cash_in, cash_out = b.get(Account.BANK, (0.0, 0.0))
        notes = []
        net_pay_payable = cr(Account.NET_PAY_PAYABLE)
        efka_payable = cr(Account.EFKA_PAYABLE)
        tax_payable = cr(Account.TAX_WITHHELD_PAYABLE)
        outstanding = round(net_pay_payable + efka_payable + tax_payable, 2)
        if payroll:
            notes.append(
                f"Payroll expense €{payroll:,.2f}, but only the net left the bank so far; "
                f"€{round(efka_payable + tax_payable, 2):,.2f} of EFKA+tax is still payable."
            )

        return FinancialStatements(
            period=self.period,
            revenue=revenue, cogs=cogs, opex=opex, payroll_expense=payroll,
            net_profit=net_profit,
            cash_in=round(cash_in, 2), cash_out=round(cash_out, 2),
            net_cash=round(cash_in - cash_out, 2),
            accounts_receivable=dr(Account.ACCOUNTS_RECEIVABLE),
            accounts_payable=cr(Account.ACCOUNTS_PAYABLE),
            bank_balance_movement=round(cash_in - cash_out, 2),
            gross_margin_pct=round((revenue - cogs) / revenue * 100, 1) if revenue else None,
            notes=notes,
        )

    def all_entries_balanced(self) -> bool:
        return all(e.is_balanced for e in self.entries)


# ── archon/documents.py ──────────────────────────────────────
"""Self-contained sample month (2026-01) for a small Greek company — a mix of
the documents a real business receives, so Archon can show classification +
correlation across types, not just one trick.

A coherent, balanced set:
  · 1 sales invoice      (revenue 5,000 + VAT 1,200 = 6,200)
  · 1 purchase invoice   (opex 1,000 + VAT 240 = 1,240)
  · 3 bank transactions  (customer pays 6,200 · we pay vendor 1,240 · payroll net 14,350)
  · 1 payroll run        (gross 23,100 → net 14,350, employer cost 28,249)

Payroll figures are realistic Greek bookkeeping (NOT a "28% gap"):
  gross 23,100  − employee EFKA 3,204 − withheld tax 5,546 = net 14,350
  employer cost = gross 23,100 + employer EFKA 5,149       = 28,249
Only 14,350 leaves the account to employees this month; 13,899 of EFKA+tax is a
payable that settles later — the real, honest cash-timing insight.
"""

SALES_INVOICE = """\
ΤΙΜΟΛΟΓΙΟ ΠΩΛΗΣΗΣ (Sales Invoice)
Αριθμός: INV-2026-014
Ημερομηνία: 12/01/2026
Πελάτης: ACME RETAIL A.E.
Καθαρή αξία: 5.000,00 EUR
ΦΠΑ 24%: 1.200,00 EUR
Σύνολο: 6.200,00 EUR
"""

PURCHASE_INVOICE = """\
ΤΙΜΟΛΟΓΙΟ ΑΓΟΡΑΣ (Purchase Invoice)
Αριθμός: SUP-8841
Ημερομηνία: 08/01/2026
Προμηθευτής: CLOUD VENDOR LTD
Καθαρή αξία: 1.000,00 EUR
ΦΠΑ 24%: 240,00 EUR
Σύνολο: 1.240,00 EUR
"""

BANK_IN = """\
ΚΙΝΗΣΗ ΛΟΓΑΡΙΑΣΜΟΥ (Bank Transaction)
Ημερομηνία: 20/01/2026
Κατεύθυνση: Εισερχόμενη (in)
Ποσό: 6.200,00 EUR
Αιτιολογία: INV-2026-014
"""

BANK_OUT_VENDOR = """\
ΚΙΝΗΣΗ ΛΟΓΑΡΙΑΣΜΟΥ (Bank Transaction)
Ημερομηνία: 15/01/2026
Κατεύθυνση: Εξερχόμενη (out)
Ποσό: 1.240,00 EUR
Αιτιολογία: SUP-8841
"""

BANK_OUT_PAYROLL = """\
ΚΙΝΗΣΗ ΛΟΓΑΡΙΑΣΜΟΥ (Bank Transaction)
Ημερομηνία: 31/01/2026
Κατεύθυνση: Εξερχόμενη (out)
Ποσό: 14.350,00 EUR
Αιτιολογία: ΜΙΣΘΟΔΟΣΙΑ ΙΑΝΟΥΑΡΙΟΥ
"""

PAYROLL = """\
ΜΙΣΘΟΔΟΣΙΑ (Payroll Run)
Περίοδος: 01/2026
Εργαζόμενοι: 2
Μικτές αποδοχές: 23.100,00 EUR
Καθαρές πληρωτέες: 14.350,00 EUR
Εργοδοτικές εισφορές ΕΦΚΑ: 5.149,00 EUR
Παρακρατούμενος φόρος (ΦΜΥ): 5.546,00 EUR
Συνολικό εργοδοτικό κόστος: 28.249,00 EUR
"""

PERIOD = "2026-01"

SAMPLE_DOCS: dict[str, str] = {
    "sales_invoice_INV-2026-014.txt": SALES_INVOICE,
    "purchase_invoice_SUP-8841.txt": PURCHASE_INVOICE,
    "bank_in_INV-2026-014.txt": BANK_IN,
    "bank_out_SUP-8841.txt": BANK_OUT_VENDOR,
    "bank_out_payroll.txt": BANK_OUT_PAYROLL,
    "payroll_2026-01.txt": PAYROLL,
}

GROUND_TRUTH = {
    "period": PERIOD,
    "revenue": 5000.00,
    "opex": 1000.00,
    "payroll_expense": 28249.00,
    "net_profit": 5000.00 - 1000.00 - 28249.00,   # -24,249.00
    "cash_in": 6200.00,
    "cash_out": 1240.00 + 14350.00,               # 15,590.00
    "net_cash": 6200.00 - 15590.00,               # -9,390.00
    "payroll_payable_remaining": 5149.00 + 3204.00 + 5546.00,  # EFKA(er+ee)+tax = 13,899
}


# ── archon/validation.py ─────────────────────────────────────
"""Cross-document consistency gates — the deterministic **eval gate**.

Booking numbers is not enough; the books are only trustworthy when the documents
*agree with each other*. Archon runs four rules (R1–R4) over the recorded month,
the same discipline the Archon family uses on Nebius and Azure, adapted to this
double-entry ledger:

  R1  the bank line that pays payroll ≈ the payroll run's net pay        (±2%)
  R2  employer cost / gross is a sane social-contribution ratio    (1.15–1.35)
  R3  the payroll payment lands on or before the period's last day
  R4  the payroll adds up: gross ≥ net + withheld tax (employee EFKA ≥ 0)

R4 is deliberately *additive* to the double-entry balance check: a balanced
journal already forces employer_cost = gross + employer_efka, so R4 instead
guards the one identity balance does **not** guarantee — that the employee-side
deductions are non-negative and internally coherent.

Every rule degrades gracefully: if the documents a rule needs are absent it is
reported as passed-but-skipped, so a partial upload is never falsely failed.
"""


from calendar import monthrange
from datetime import date, datetime




# Employer cost as a multiple of gross pay. Greek employer EFKA is ~22%, so the
# ratio sits near 1.22; the band tolerates other jurisdictions/rounding.
_EMPLOYER_COST_BAND = (1.15, 1.35)
_BANK_TOLERANCE = 0.02  # ±2%


def _payroll(docs: list[Document]) -> Document | None:
    return next((d for d in docs if d.doc_type == DocType.PAYROLL), None)


def _payroll_bank_line(docs: list[Document]) -> Document | None:
    for d in docs:
        if d.doc_type == DocType.BANK_TRANSACTION and any(
            k in (d.reference or "") for k in _PAYROLL_KEYS
        ):
            return d
    return None


def _parse_date(value: str | None) -> date | None:
    if not value:
        return None
    value = value.strip()
    for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%d-%m-%Y"):
        try:
            return datetime.strptime(value, fmt).date()
        except ValueError:
            continue
    return None


def _skip(rule: str) -> ValidationResult:
    return ValidationResult(rule=rule, passed=True, severity="info",
                            message="Skipped — required documents absent.")


# ── R1 ──────────────────────────────────────────────────────────────────────
def r1_bank_matches_payroll_net(docs: list[Document]) -> ValidationResult:
    rule = "R1: bank payroll line ≈ payroll net pay (±2%)"
    payroll = _payroll(docs)
    bank = _payroll_bank_line(docs)
    if not payroll or not bank:
        return _skip(rule)
    net = payroll.net_pay_total or 0.0
    paid = bank.net_amount or 0.0
    if net == 0:
        return ValidationResult(rule, False, "error", "Payroll net pay is zero.")
    deviation = abs(paid - net) / net
    passed = deviation <= _BANK_TOLERANCE
    return ValidationResult(
        rule, passed, "info" if passed else "error",
        f"Bank paid €{paid:,.2f} vs payroll net €{net:,.2f} ({deviation * 100:.1f}% deviation)",
    )


# ── R2 ──────────────────────────────────────────────────────────────────────
def r2_employer_cost_ratio(docs: list[Document]) -> ValidationResult:
    lo, hi = _EMPLOYER_COST_BAND
    rule = f"R2: employer cost / gross in [{lo}, {hi}]"
    payroll = _payroll(docs)
    if not payroll or not payroll.gross_amount or not payroll.employer_cost_total:
        return _skip(rule)
    gross = payroll.gross_amount
    ratio = payroll.employer_cost_total / gross
    passed = lo <= ratio <= hi
    return ValidationResult(
        rule, passed, "info" if passed else "warning",
        f"employer_cost/gross = {ratio:.3f} (expected {lo}–{hi})",
    )


# ── R3 ──────────────────────────────────────────────────────────────────────
def r3_payment_within_period(docs: list[Document], period: str) -> ValidationResult:
    rule = "R3: payroll payment date ≤ period end"
    bank = _payroll_bank_line(docs)
    paid_on = _parse_date(bank.date if bank else None)
    if not bank or not paid_on or not period or len(period) < 7:
        return _skip(rule)
    try:
        year, month = int(period[:4]), int(period[5:7])
        last_day = date(year, month, monthrange(year, month)[1])
    except (ValueError, IndexError) as exc:
        return ValidationResult(rule, False, "warning", f"Bad period '{period}': {exc}")
    passed = paid_on <= last_day
    return ValidationResult(
        rule, passed, "info" if passed else "warning",
        f"Paid {paid_on.isoformat()} vs period end {last_day.isoformat()}",
    )


# ── R4 ──────────────────────────────────────────────────────────────────────
def r4_payroll_identity(docs: list[Document]) -> ValidationResult:
    rule = "R4: gross ≥ net + withheld tax (employee EFKA ≥ 0)"
    payroll = _payroll(docs)
    if not payroll or not payroll.gross_amount:
        return _skip(rule)
    gross = payroll.gross_amount
    net = payroll.net_pay_total or 0.0
    tax = payroll.tax_withheld_total or 0.0
    employee_efka = round(gross - net - tax, 2)
    passed = employee_efka >= 0
    return ValidationResult(
        rule, passed, "info" if passed else "error",
        f"implied employee EFKA = €{employee_efka:,.2f} "
        f"(gross {gross:,.2f} − net {net:,.2f} − tax {tax:,.2f})",
    )


def validate(ledger) -> list[ValidationResult]:
    """Run every gate over a ledger's documents. Pure/deterministic — no LLM."""
    docs = ledger.documents
    return [
        r1_bank_matches_payroll_net(docs),
        r2_employer_cost_ratio(docs),
        r3_payment_within_period(docs, ledger.period),
        r4_payroll_identity(docs),
    ]


def all_passed(results: list[ValidationResult]) -> bool:
    return all(r.passed for r in results)


def summary(results: list[ValidationResult]) -> str:
    passed = sum(1 for r in results if r.passed)
    return f"{passed}/{len(results)} validation gates passed"


# ── archon/narrator.py ───────────────────────────────────────
"""Executive narration over the computed books.

Two layers, same facts:

* `narrate()` is deterministic — it turns the statements + validation gates into
  a short CFO-style paragraph with **zero** model calls, so the notebook and the
  tests always produce a summary offline.
* The ADK+Gemini agent (`agent.py`) and the analysis pipeline (`pipeline.py`)
  wrap the *same* numbers and let Gemini phrase them; the model never invents a
  figure, it only narrates what the deterministic engine computed.
"""





def narrate(stmt: FinancialStatements, validation: list[ValidationResult] | None = None) -> str:
    """A concise, deterministic executive summary of one month's books."""
    verdict = "profit" if stmt.net_profit >= 0 else "loss"
    parts = [
        f"In {stmt.period} the business booked €{stmt.revenue:,.2f} of revenue against "
        f"€{stmt.opex + stmt.payroll_expense:,.2f} of operating and payroll expense, "
        f"a net {verdict} of €{abs(stmt.net_profit):,.2f}."
    ]

    if stmt.payroll_expense and stmt.payroll_expense > stmt.cash_out:
        held_back = round(stmt.payroll_expense - (stmt.cash_out - stmt.opex), 2)
        parts.append(
            f"Cash tells a different story: only €{stmt.cash_out:,.2f} actually left the "
            f"account, because EFKA and withheld tax are payables that settle later — the "
            f"P&L expense of €{stmt.payroll_expense:,.2f} runs ahead of the cash."
        )

    parts.append(
        f"The month closes with €{stmt.net_cash:,.2f} net cash movement, "
        f"AR €{stmt.accounts_receivable:,.2f} and AP €{stmt.accounts_payable:,.2f}."
    )

    if validation:
        passed = sum(1 for r in validation if r.passed)
        failed = [r for r in validation if not r.passed]
        if failed:
            parts.append(
                f"Validation: {passed}/{len(validation)} gates passed — review "
                + "; ".join(r.rule.split(':')[0] for r in failed) + "."
            )
        else:
            parts.append(
                f"All {len(validation)} cross-document validation gates passed, so the "
                f"books reconcile end to end."
            )

    return " ".join(parts)


def facts_sheet(stmt: FinancialStatements, validation: list[ValidationResult] | None = None) -> str:
    """A compact, model-friendly fact sheet the LLM narration layer reads from.

    Kept deterministic so the pipeline's Gemini turn only ever *phrases* figures
    it was handed — it can never invent one.
    """
    lines = [
        f"Period: {stmt.period}",
        f"Revenue: {stmt.revenue:.2f}",
        f"Operating expense: {stmt.opex:.2f}",
        f"Payroll expense (true employer cost): {stmt.payroll_expense:.2f}",
        f"Net profit: {stmt.net_profit:.2f}",
        f"Cash in: {stmt.cash_in:.2f}",
        f"Cash out: {stmt.cash_out:.2f}",
        f"Net cash: {stmt.net_cash:.2f}",
        f"Accounts receivable: {stmt.accounts_receivable:.2f}",
        f"Accounts payable: {stmt.accounts_payable:.2f}",
    ]
    for n in stmt.notes:
        lines.append(f"Note: {n}")
    if validation:
        for r in validation:
            lines.append(f"{'PASS' if r.passed else 'FAIL'} {r.rule} — {r.message}")
    return "\n".join(lines)


In [ ]:
# Ingest a mixed month, keep the books, show the result
led = Ledger(period=GROUND_TRUTH["period"], company="Reflective IKE")
for name, text in SAMPLE_DOCS.items():
    doc = extract_document(text, source_file=name, period=led.period)
    led.add(doc)
    print(f"classified {doc.doc_type.value:<17} <- {name}")

s = led.statements()
print("\n=== JOURNAL (double-entry) ===")
for e in led.entries:
    print(("  balanced  " if e.is_balanced else "  UNBALANCED ") + e.memo)

print("\n=== RECONCILIATION ===")
for m in led.reconcile():
    print(f"  {'OK ' if m.matched else 'X  '}{m.bank_ref:<24} EUR {m.amount:>10,.2f}  {m.note}")

print("\n=== P&L ===")
print(f"  Revenue            EUR {s.revenue:>12,.2f}")
print(f"  Operating expenses EUR {s.opex:>12,.2f}")
print(f"  Payroll expense    EUR {s.payroll_expense:>12,.2f}")
print(f"  Net profit         EUR {s.net_profit:>12,.2f}")
print("\n=== CASH ===")
print(f"  In  EUR {s.cash_in:>12,.2f}   Out EUR {s.cash_out:>12,.2f}   Net EUR {s.net_cash:>12,.2f}")
print(f"  AR EUR {s.accounts_receivable:,.2f}   AP EUR {s.accounts_payable:,.2f}")
for n in s.notes:
    print("  note:", n)

print("\n=== VALIDATION (R1-R4 eval gate) ===")
checks = validate(led)
for r in checks:
    print(f"  {'PASS' if r.passed else 'FAIL'} {r.rule}")
    print(f"        {r.message}")

print("\n=== EXECUTIVE SUMMARY (deterministic) ===")
print(" ", narrate(s, checks))

assert led.all_entries_balanced()
assert s.net_profit == GROUND_TRUTH["net_profit"]
assert all(r.passed for r in checks)
print("\nSelf-check passed: every entry balances; R1-R4 gates hold; figures match ground truth.")

## The analysis pipeline — an ADK `SequentialAgent` (multi-agent)

The Archon family runs a *chain* of specialised agents. Here three sub-agents run
in order — **reconciler → validator → narrator** — handing state forward via
`output_key`, exactly the course's multi-agent pattern. The deterministic engine
computes the numbers; the pipeline narrates them.

The cell below runs the **real** ADK `SequentialAgent` with **scripted models**,
so it executes **offline with no API key** — a faithful demonstration of the
concept that still runs top-to-bottom in the published notebook.

In [ ]:
# Ensure google-adk is available (Kaggle has internet enabled).
try:
    import google.adk  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-adk"], check=False)

try:
    from typing import AsyncGenerator
    from google.adk.agents import LlmAgent, SequentialAgent
    from google.adk.models import BaseLlm
    from google.adk.models.llm_response import LlmResponse
    from google.adk.runners import InMemoryRunner
    from google.genai import types
    HAVE_ADK = True
except Exception as exc:
    print("google-adk unavailable, skipping the SequentialAgent demo:", exc)
    HAVE_ADK = False

if HAVE_ADK:
    class ScriptedText(BaseLlm):
        """A model that returns one fixed line — lets the SequentialAgent run offline."""
        reply: str = "ok"
        def __init__(self, reply, **kw):
            super().__init__(model="scripted", **kw); object.__setattr__(self, "reply", reply)
        async def generate_content_async(self, req, stream=False) -> "AsyncGenerator[LlmResponse, None]":
            yield LlmResponse(content=types.Content(role="model", parts=[types.Part(text=self.reply)]))

    # Deterministic engine first: compute the books + gates, build the fact sheet.
    led = Ledger(period=GROUND_TRUTH["period"], company="Reflective IKE")
    for name, text in SAMPLE_DOCS.items():
        led.add(extract_document(text, source_file=name, period=led.period))
    facts = facts_sheet(led.statements(), validate(led))

    reconciler = LlmAgent(name="reconciler", model=ScriptedText("All bank lines reconcile."),
                          instruction="Reconcile the books.", output_key="reconciliation")
    validator = LlmAgent(name="validator", model=ScriptedText("R1-R4 all pass."),
                         instruction="Validate {reconciliation}.", output_key="validation")
    narrator = LlmAgent(name="narrator", model=ScriptedText("Net loss; payroll expense runs ahead of cash."),
                        instruction="Narrate {validation}.", output_key="summary")
    pipeline = SequentialAgent(name="archon_analysis", sub_agents=[reconciler, validator, narrator])

    runner = InMemoryRunner(agent=pipeline, app_name="archon-analysis")
    try:
        runner.session_service.create_session_sync(app_name="archon-analysis", user_id="owner", session_id="a1")
    except AttributeError:
        import asyncio; asyncio.get_event_loop().run_until_complete(
            runner.session_service.create_session(app_name="archon-analysis", user_id="owner", session_id="a1"))
    for ev in runner.run(user_id="owner", session_id="a1", new_message=types.Content(
            role="user", parts=[types.Part(text="Analyse:\n"+facts)])):
        if ev.content and ev.content.parts and ev.content.parts[0].text:
            print(f"  [{ev.author}] {ev.content.parts[0].text}")
    print("\nThree agents ran in sequence, each writing to shared state (output_key).")

## The conversational agent (Google ADK + Gemini)

Above is the deterministic engine. Archon's **agent** wraps it: the owner feeds
documents in across turns, and Gemini decides when to call `record_document`,
`reconcile_bank`, and `get_books` — with the books held in **session memory**.

Add a **`GOOGLE_API_KEY`** secret (Kaggle → Add-ons → Secrets) and run the cell
below. Without a key, the deterministic books above are identical — the agent is
the conversational layer, not the source of the numbers.

In [ ]:
import os
# On Kaggle, the GOOGLE_API_KEY lives in the notebook's Secrets (Add-ons -> Secrets),
# not the environment — pull it into os.environ so the live agent can run.
if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["GOOGLE_API_KEY"] = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    except Exception:
        pass
try:
    from google.adk.agents import Agent
    from google.adk.runners import InMemoryRunner
    from google.genai import types
    HAVE_ADK = bool(os.environ.get("GOOGLE_API_KEY"))
except Exception:
    HAVE_ADK = False

if not HAVE_ADK:
    print("google-adk + GOOGLE_API_KEY not configured — skipping the live agent.")
    print("The deterministic books above are the same numbers the agent reports.")
else:
    session_ledger = Ledger(period="2026-01", company="Reflective IKE")

    def record_document(doc_text: str, source_file: str = "doc.txt") -> dict:
        """Classify and post one business document to the books."""
        doc = extract_document(doc_text, source_file=source_file, period=session_ledger.period)
        e = session_ledger.add(doc)
        return {"classified_as": doc.doc_type.value, "memo": e.memo, "balanced": e.is_balanced}

    def reconcile_bank() -> dict:
        """Match every bank line to the invoice or payroll it settles."""
        return {"matches": [
            {"bank_ref": m.bank_ref, "matched": m.matched, "amount": m.amount, "note": m.note}
            for m in session_ledger.reconcile()
        ]}

    def validate_books() -> dict:
        """Run the R1-R4 cross-document consistency gates over the books."""
        return {"gates": [{"rule": r.rule, "passed": r.passed, "message": r.message}
                          for r in validate(session_ledger)]}

    def get_books() -> dict:
        """Return the period P&L, cash view, and AR/AP."""
        s = session_ledger.statements()
        return {"revenue": s.revenue, "opex": s.opex, "payroll_expense": s.payroll_expense,
                "net_profit": s.net_profit, "cash_in": s.cash_in, "cash_out": s.cash_out,
                "net_cash": s.net_cash, "notes": s.notes}

    agent = Agent(name="archon_bookkeeper", model="gemini-2.5-flash",
                  instruction="You are Archon, an autonomous bookkeeper. Call record_document for "
                  "each document the user provides, reconcile_bank to match bank lines to invoices "
                  "and payroll, validate_books to run the R1-R4 gates, and get_books when asked "
                  "about money. Payroll "
                  "EXPENSE exceeds the net that leaves the bank; the difference (EFKA+tax) is a "
                  "payable that settles later — surface that. Never invent figures.",
                  tools=[record_document, reconcile_bank, validate_books, get_books])
    runner = InMemoryRunner(agent=agent, app_name="archon")
    try:
        runner.session_service.create_session_sync(app_name="archon", user_id="owner", session_id="s1")
    except AttributeError:
        import asyncio; asyncio.get_event_loop().run_until_complete(
            runner.session_service.create_session(app_name="archon", user_id="owner", session_id="s1"))

    for _, text in list(SAMPLE_DOCS.items()):
        list(runner.run(user_id="owner", session_id="s1",
             new_message=types.Content(role="user", parts=[types.Part(text="Record:\n"+text)])))
    for ev in runner.run(user_id="owner", session_id="s1", new_message=types.Content(
            role="user", parts=[types.Part(text="Show the P&L and cash position. What did payroll "
            "really cost vs what left the account?")])):
        if ev.is_final_response() and ev.content and ev.content.parts:
            print(ev.content.parts[0].text)

## How it maps to the course
- **Tools / function calling** — `record_document`, `reconcile_bank`, `validate_books`, `get_books`
- **Multi-agent (`SequentialAgent`)** — reconciler → validator → narrator, state via `output_key`
- **Memory / state** — the agent's ledger accumulates documents across turns
- **Evaluation / guardrails** — every entry must balance; the R1–R4 gates must hold
- **Vibe coding** — each module is a one-paragraph spec, prompted against ground-truth tests

**Code:** https://github.com/upgradedev/archon-gcp · MIT · also runs on Nebius and Azure.
Full test pyramid (unit → integration → e2e, all offline) + security CI (gitleaks · CodeQL · pip-audit).